# Day 3 — Dataset Preparation, Preprocessing, and Baseline Model

**Objective:** Build a reproducible supervised-learning workflow for predicting customer churn.

This notebook loads the source CSV independently, applies the documented `TotalCharges` cleaning assumption, separates features and target, splits the data, builds a preprocessing/model pipeline, and evaluates a Logistic Regression baseline.

**Important evaluation note:** The 0.30 threshold experiment below was explored on the test set for learning. Those results are exploratory; final threshold selection should use validation data or cross-validation, followed by a final evaluation on an untouched test set.

## 1. Imports and Load Data

This notebook must run independently; it does not depend on variables created in `01_data_inspection.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

df = pd.read_csv("../data/telco-customer-churn.csv")
print("Dataset shape before cleaning:", df.shape)

## 2. Apply the Documented Cleaning Step

In the inspection notebook, 11 `TotalCharges` entries were found to contain whitespace rather than ordinary null values. These were associated with zero-tenure customers. We provisionally replace blank values with zero, as documented in the EDA notebook; validate this assumption against the dataset definition before production use.

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].str.strip().replace("", pd.NA),
    errors="raise"
)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Missing values after cleaning:", df.isnull().sum().sum())
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Dataset shape after cleaning:", df.shape)

## 3. Define Features (`X`) and Target (`y`)

`Churn` is the outcome we want to predict, so it must not appear among the input features. `customerID` is an identifier and is excluded from the initial model inputs.

In [ ]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution (%):")
print(y.value_counts(normalize=True).mul(100).round(2))

## 4. Train/Test Split

We reserve 20% of the records for testing. `stratify=y` preserves the churn/non-churn proportions approximately in both partitions. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining target proportions (%):")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTest target proportions (%):")
print(y_test.value_counts(normalize=True).mul(100).round(2))

## 5. Identify Feature Groups

We use data types to identify the initial groups, then explicitly treat `SeniorCitizen` as a categorical indicator. It is stored as 0/1, but those values represent categories rather than a continuous measurement. Treating it as numeric could also be defensible; this notebook makes the chosen treatment explicit.

In [ ]:
numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numerical_features.remove("SeniorCitizen")
categorical_features.append("SeniorCitizen")

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

## 6. Build the Preprocessor

`StandardScaler` learns each numerical feature's mean and standard deviation from training data. `OneHotEncoder` converts categorical values into indicator columns. `handle_unknown="ignore"` prevents an unseen category from causing an error during transformation.

We will fit this preprocessor through the pipeline, using training data only.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## 7. Build and Train the Baseline Pipeline

The pipeline combines preprocessing and Logistic Regression. Passing the original `X_train` to `.fit()` lets the pipeline fit the transformations on training data and then train the classifier on the transformed features.

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

model.fit(X_train, y_train)

## 8. Generate Test Predictions

`predict()` applies the fitted preprocessing and returns class predictions. The test labels are not used to fit the model.

In [ ]:
y_pred = model.predict(X_test)

print("First 10 predictions:", y_pred[:10])
print("First 10 actual labels:", y_test.iloc[:10].to_list())

## 9. Baseline Evaluation at the Default Threshold

For this binary classification problem, we treat `Yes` (churn) as the positive class.

In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, pos_label='Yes'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, pos_label='Yes'):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred, pos_label='Yes'):.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["No", "Yes"])
print(cm)

cm_display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No churn", "Churn"]
)
cm_display.plot()
plt.show()

## 10. Exploratory Threshold Experiment (Not Final Selection)

Logistic Regression's `predict_proba()` estimates the probability of each class. Verify the class order before selecting the positive-class probability column. Here, the experiment uses the test set only to reproduce the earlier learning exercise; do not use these test results to claim a final optimized threshold.

In [ ]:
print("Classifier class order:", model.named_steps["classifier"].classes_)

y_prob = model.predict_proba(X_test)
print("Probability array shape:", y_prob.shape)
print("First 10 churn probabilities (%):", (y_prob[:10, 1] * 100).round(2))

In [ ]:
y_pred_30 = np.where(y_prob[:, 1] >= 0.30, "Yes", "No")

print(f"Precision at threshold 0.30: {precision_score(y_test, y_pred_30, pos_label='Yes'):.4f}")
print(f"Recall at threshold 0.30:    {recall_score(y_test, y_pred_30, pos_label='Yes'):.4f}")
print(f"F1-score at threshold 0.30:   {f1_score(y_test, y_pred_30, pos_label='Yes'):.4f}")

cm_30 = confusion_matrix(y_test, y_pred_30, labels=["No", "Yes"])
print("Confusion matrix at threshold 0.30:")
print(cm_30)

## 11. Interpretation and Limitations

The previous exploratory run reported:

- Threshold 0.50: precision 65.72%, recall 55.88%, F1-score 60.40%.
- Threshold 0.30: precision 51.93%, recall 75.40%, F1-score 61.50%.

Lowering the threshold identified more churners but also produced more false positives. The 0.30 threshold is **not** the final selected threshold.

**Evaluation limitation:** We used the test set to explore the threshold. For rigorous model selection, introduce a validation split or cross-validation, choose the threshold without using the final test set, and evaluate the selected approach on an untouched test set at the end.

**Next steps:** Conduct systematic error analysis, compare models and metrics using a validation strategy, and consider business costs when selecting a threshold.